In [ ]:
raise Exception("")

FACILITIES = ['MIA', 'TPA', 'RSW']

import json, requests, time
from datetime import datetime, timezone

def parse_facility_data(facility, timeout=2):
    url = f"https://dstars.graiani.com/dstars/{facility}/updates"

    data = []   
    with requests.get(url, stream=True, timeout=timeout * 1.5, headers={'User-Agent': 'Mozilla/5.0'}) as r:
        r.raise_for_status()
        t0 = time.time()
        for line in r.iter_lines():
            if time.time() - t0 > timeout:
                break
            
            if line:
                data.append(json.loads(line.decode("utf-8")))

    owners = []
    for l in data:
        if 'Owner' in l:
            owner = l['Owner']
            if len(owner) == 2 and owner not in owners:
                owners.append(owner)

    
    ct = datetime.now(timezone.utc).strftime("%H%M")
    return ct + '\t' + ' '.join(sorted(owners))


while True:
    for facility in FACILITIES:
        ownership = parse_facility_data(facility, timeout=1)
        with open(f"{facility.lower()}_track_ownership.txt", "a") as f:
            f.write(ownership + '\n')
            print(facility + '\t' + ownership)
    
    time.sleep(300)